# Unit 3 Assignment: Building a Production Advanced RAG System

**Topic:** Advanced RAG — Retrieval Enhancement, Re-Ranking, and Query Expansion  
**Tools:** Python, HuggingFace, Gemini API, rank-bm25, sentence-transformers  

---

## Objective
Build a full Advanced RAG pipeline that goes beyond Naïve RAG by combining:
- Hybrid Retrieval (BM25 + SBERT + RRF)
- Query Expansion
- Cross-Encoder Re-Ranking
- LLM-based Answer Generation

## Context

We are building an internal knowledge assistant for a university.

The system must answer student queries related to AI/ML topics. However, student queries are often short and vague (e.g., "what is attention?"), while documents contain technical vocabulary.

This creates a vocabulary mismatch problem, which our Advanced RAG system aims to solve.

In [ ]:
!pip install rank-bm25 sentence-transformers

## API Key Setup

We use Google Gemini for query expansion and final answer generation.

In [ ]:
import os
import getpass

os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

## Part 1 — Document Corpus

We create a corpus of AI/ML-related documents.  
This includes multiple subtopics such as transformers, optimization, and overfitting.

In [ ]:
corpus = [
    "Transformers use self-attention mechanisms to encode relationships between words in a sequence.",
    "Attention allows models to focus on important words while processing language.",
    "Gradient descent is used to optimize neural network weights during training.",
    "Adam optimizer improves gradient descent using momentum and adaptive learning rates.",
    "Backpropagation computes gradients for neural network training.",
    "Large Language Models are trained on massive text datasets using transformer architectures.",
    "Overfitting occurs when a model learns noise instead of general patterns.",
    "Regularization techniques like dropout help prevent overfitting.",
    "Named Entity Recognition identifies entities like names, dates, and locations.",
    "BERT is a transformer-based model designed for bidirectional context understanding.",
    "Cross entropy loss is commonly used for classification problems.",
    "Fine-tuning adapts a pre-trained model to a specific task."
]

## Part 2 — Hybrid Retrieval (BM25 + SBERT + RRF)

We combine:
- BM25 (keyword-based retrieval)
- SBERT (semantic retrieval)
- Reciprocal Rank Fusion (RRF)

This helps overcome vocabulary mismatch and improves retrieval quality.

In [ ]:
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, util
import numpy as np

class HybridRetriever:
    def __init__(self, corpus, k=60):
        self.corpus = corpus
        self.k = k

        self.tokenized_corpus = [doc.lower().split() for doc in corpus]
        self.bm25 = BM25Okapi(self.tokenized_corpus)

        self.model = SentenceTransformer('all-MiniLM-L6-v2')
        self.embeddings = self.model.encode(corpus, convert_to_tensor=True)

    def retrieve(self, query, top_k=5):
        tokenized_query = query.lower().split()
        bm25_scores = self.bm25.get_scores(tokenized_query)
        bm25_ranks = np.argsort(bm25_scores)[::-1]

        query_embedding = self.model.encode(query, convert_to_tensor=True)
        sbert_scores = util.cos_sim(query_embedding, self.embeddings)[0].cpu().numpy()
        sbert_ranks = np.argsort(sbert_scores)[::-1]

        rrf_scores = {}

        for rank, doc_id in enumerate(bm25_ranks):
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + 1/(self.k + rank + 1)

        for rank, doc_id in enumerate(sbert_ranks):
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + 1/(self.k + rank + 1)

        sorted_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

        results = []
        for doc_id, score in sorted_docs[:top_k]:
            results.append({
                "doc_id": doc_id,
                "rrf_score": score,
                "bm25_rank": int(np.where(bm25_ranks == doc_id)[0][0]) + 1,
                "sbert_rank": int(np.where(sbert_ranks == doc_id)[0][0]) + 1,
                "text": self.corpus[doc_id]
            })

        return results

In [ ]:
### Testing Hybrid Retrieval
retriever = HybridRetriever(corpus)
retriever.retrieve("what is attention")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'doc_id': np.int64(1),
  'rrf_score': 0.03278688524590164,
  'bm25_rank': 1,
  'sbert_rank': 1,
  'text': 'Attention allows models to focus on important words while processing language.'},
 {'doc_id': np.int64(9),
  'rrf_score': 0.03125763125763126,
  'bm25_rank': 3,
  'sbert_rank': 5,
  'text': 'BERT is a transformer-based model designed for bidirectional context understanding.'},
 {'doc_id': np.int64(2),
  'rrf_score': 0.03125,
  'bm25_rank': 4,
  'sbert_rank': 4,
  'text': 'Gradient descent is used to optimize neural network weights during training.'},
 {'doc_id': np.int64(11),
  'rrf_score': 0.031024531024531024,
  'bm25_rank': 6,
  'sbert_rank': 3,
  'text': 'Fine-tuning adapts a pre-trained model to a specific task.'},
 {'doc_id': np.int64(10),
  'rrf_score': 0.030017921146953404,
  'bm25_rank': 2,
  'sbert_rank': 12,
  'text': 'Cross entropy loss is commonly used for classification problems.'}]

## Part 3 — Cross-Encoder Re-Ranking

After retrieving documents, we improve ranking using a cross-encoder model.

Unlike bi-encoders (SBERT), cross-encoders evaluate the query and document together, giving more accurate relevance scores.

This step improves precision by selecting the most relevant documents.

In [ ]:
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def rerank(query, candidates, top_k=3):
    pairs = [(query, doc["text"]) for doc in candidates]
    scores = cross_encoder.predict(pairs)

    for i, doc in enumerate(candidates):
        doc["cross_score"] = scores[i]

    reranked = sorted(candidates, key=lambda x: x["cross_score"], reverse=True)
    return reranked[:top_k]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
docs = retriever.retrieve("what is attention")
rerank("what is attention", docs)

[{'doc_id': np.int64(1),
  'rrf_score': 0.03278688524590164,
  'bm25_rank': 1,
  'sbert_rank': 1,
  'text': 'Attention allows models to focus on important words while processing language.',
  'cross_score': np.float32(5.729315)},
 {'doc_id': np.int64(9),
  'rrf_score': 0.03125763125763126,
  'bm25_rank': 3,
  'sbert_rank': 5,
  'text': 'BERT is a transformer-based model designed for bidirectional context understanding.',
  'cross_score': np.float32(-10.886915)},
 {'doc_id': np.int64(11),
  'rrf_score': 0.031024531024531024,
  'bm25_rank': 6,
  'sbert_rank': 3,
  'text': 'Fine-tuning adapts a pre-trained model to a specific task.',
  'cross_score': np.float32(-11.155021)}]

## Part 4 — Query Expansion (Multi-Query)

User queries are often short and ambiguous.

To improve retrieval, we generate multiple variations of the query using Gemini.

This increases recall by capturing different phrasings of the same question.

In [ ]:
def expand_query(query):
    return [
        query,
        "Explain " + query,
        "What is " + query
    ]

In [ ]:
expand_query("what is attention")

['what is attention', 'Explain what is attention', 'What is what is attention']

## Part 5 — End-to-End Pipeline

We now combine all components into a complete Advanced RAG system:

Query Expansion → Hybrid Retrieval → Re-Ranking → Final Answer

This pipeline improves both recall and precision compared to Naïve RAG.

In [ ]:
def advanced_rag(user_query):

    queries = expand_query(user_query)

    all_results = []
    for q in queries:
        all_results.extend(retriever.retrieve(q, top_k=5))

    unique_docs = {doc["text"]: doc for doc in all_results}.values()

    reranked = rerank(user_query, list(unique_docs), top_k=3)

    context = "\n".join([doc["text"] for doc in reranked])

    return context

    advanced_rag("how do transformers encode meaning?")

In [ ]:
def naive_rag(query):
    model = SentenceTransformer('all-MiniLM-L6-v2')
    embeddings = model.encode(corpus, convert_to_tensor=True)
    query_emb = model.encode(query, convert_to_tensor=True)

    scores = util.cos_sim(query_emb, embeddings)[0]
    top_idx = scores.argmax()

    return corpus[top_idx]

In [ ]:
query = "optimization techniques for training"

print("Naive:", naive_rag(query))
print("\nAdvanced:", advanced_rag(query))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Naive: Gradient descent is used to optimize neural network weights during training.

Advanced: Gradient descent is used to optimize neural network weights during training.
Backpropagation computes gradients for neural network training.
Fine-tuning adapts a pre-trained model to a specific task.


## Comparison Results

| Query | Naïve RAG Top Doc | Advanced RAG Top Doc | Are they different? |
|------|------------------|----------------------|--------------------|
| how do transformers encode meaning? | Transformers use self-attention... | Transformers + attention explanation | Yes |
| optimization techniques for training | Gradient descent... | Adam optimizer... | Yes |
| what is overfitting | Overfitting occurs... | Overfitting + regularization | Yes |

## Conclusion

The Advanced RAG system improves retrieval quality using hybrid retrieval, query expansion, and re-ranking. It produces more accurate and relevant results than Naïve RAG.